# Speeding up training
You have probably noticed how slow training can get when running for many timesteps. This notebook explains how to transition from single-step to multi-step mode and how this can help you make the most of your hardware. In some cases, we have achieved substantial speedups, though the actual gains will depend on how much additional parallelization your setup can leverage.

Multi-step mode is different from single-step mode by how it passes tensors around and how the computations are done. Where single-step mode takes input for a single timestep, multi-step mode takes input from many timesteps. Computations are applied to all these timesteps at once, if possible.

## The data, network, standard setup
We prepare a fully-connected network for SHD, using the same neuron structure as the previous notebooks: an `IdentitySpikeFunction` encodes the input, hidden neurons receive and return kernel components and spikes, and the output neuron returns a continuous signal.

For multi-step execution, `forward` takes `step_mode` and passes it to the input spike function and every neuron layer. We preserve both the time and batch dimensions when flattening multi-step inputs. Input spike counts are converted to the model's floating-point dtype so the encoder can retain fractional kernel values.

In [1]:
import torch
import torch.nn as nn
import tonic
from torch.utils.data import DataLoader
from tqdm import tqdm
from spikinggamma.neuron import SpikingGammaNeuron
from spikinggamma.spike.sigma_delta import SigmaDeltaSpikeFunction
from spikinggamma.spike.identity import IdentitySpikeFunction
from spikinggamma.dendrite.split import SplitDendriticFunction
from spikinggamma.dendrite.synapse.linear import LinearPerNeuron
from spikinggamma.signal.default import DefaultSignalFunction
from spikinggamma.spike.base import SpikeConfig
from spikinggamma.utils.enums import StepMode
from spikinggamma.utils.helpers import MultiStepCrossEntropyLoss
from spikinggamma.spike.kernel.gamma import GammaKernelFunction, compute_linspace_bucket_decay_rates
from spikinggamma.utils.datasets.shd import get_shd_dataset
from pathlib import Path


project_dir = Path.cwd().parent
data_path = project_dir / "data"
cache_path = project_dir / "cache"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Data
train_dataset, test_dataset, input_shape, n_outputs, T = get_shd_dataset(data_path=data_path, cache_path=cache_path, timestep_size=3600)

# Hyperparameters
n_kernels = 10
decay_factor = 0.1
spike_config = SpikeConfig(
    base_spike_threshold=0.3,
    use_dynamic_threshold=True
)
bucket_decay_rates = compute_linspace_bucket_decay_rates(n_kernels, decay_factor)

# A simple fully connected network with 3 hidden layers
class SpikingGammaNet(nn.Module):
    def __init__(self, 
                 input_shape, 
                 n_outputs,
                 n_kernels,
                 bucket_decay_rates,
                 spike_config):
        super().__init__()
        layer1_shape_out = (256,)
        layer2_shape_out = (256,)
        layer3_shape_out = (256,)
        layer4_shape_out = (n_outputs,)

        self.input_fn = IdentitySpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config)
        self.hid1 = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                layer1_shape_out,
                n_kernels=n_kernels,
                synaptic_function=LinearPerNeuron(input_shape[0], layer1_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(pre_act_fn=nn.LayerNorm(layer1_shape_out)),
            spike_fn=SigmaDeltaSpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config),
        )

        self.hid2 = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                layer2_shape_out,
                n_kernels=n_kernels,
                synaptic_function=LinearPerNeuron(layer1_shape_out[0], layer2_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(pre_act_fn=nn.LayerNorm(layer2_shape_out)),
            spike_fn=SigmaDeltaSpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config),
        )

        self.hid3 = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                layer3_shape_out,
                n_kernels=n_kernels,
                synaptic_function=LinearPerNeuron(layer2_shape_out[0], layer3_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(pre_act_fn=nn.LayerNorm(layer3_shape_out)),
            spike_fn=SigmaDeltaSpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config),
        )

        self.out = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                layer4_shape_out,
                n_kernels=n_kernels,
                synaptic_function=LinearPerNeuron(layer3_shape_out[0], layer4_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(pre_act_fn=nn.LayerNorm(layer4_shape_out))
        )

    def reset(self):
        self.input_fn.reset()
        self.hid1.reset()
        self.hid2.reset()
        self.hid3.reset()
        self.out.reset()

    def detach(self):
        self.input_fn.detach()
        self.hid1.detach()
        self.hid2.detach()
        self.hid3.detach()
        self.out.detach()

    def forward(self, s_in, step_mode: StepMode):
        s_in = s_in.flatten(start_dim=1 if step_mode == StepMode.SINGLE else 2)
        s_in = s_in.to(dtype=self.input_fn.spike_threshold.dtype)
        y_hat_k_in, s_in = self.input_fn(s_in, step_mode=step_mode)
        y_hat_k_hid1, s_hid1 = self.hid1(y_hat_k_in, step_mode=step_mode)
        y_hat_k_hid2, s_hid2 = self.hid2(y_hat_k_hid1, step_mode=step_mode)
        y_hat_k_hid3, s_hid3 = self.hid3(y_hat_k_hid2, step_mode=step_mode)
        y_out = self.out(y_hat_k_hid3, step_mode=step_mode)
        return y_out

cuda


## Training in multi-step mode
With the network ready, we can continue with training. We need to make two important changes:
1. Instead of `nn.CrossEntropyLoss`, we use `MultiStepCrossEntropyLoss`. This can take output from many timesteps and then compute the loss per timestep, just like for single-step mode.
2. We do not have a for loop over timesteps. Instead we pass the whole data tensor to the model at once.

In [2]:
train_batch_size = 32
test_batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(batch_first=False))
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(batch_first=False))
model = SpikingGammaNet(input_shape, n_outputs, n_kernels, bucket_decay_rates, spike_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = MultiStepCrossEntropyLoss()

We first show the *stateless* example. This means that the model will not preserve a state in between forward calls at all.

In [3]:
step_mode = StepMode.MULTI_STATELESS
model.train()
n_epochs = 1

for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        y = model(data, step_mode=step_mode)
        loss = loss_fn(y, labels)
        loss.backward()

        optimizer.step()
        pbar.set_description(f"Loss: {loss.item():.4f}")

Loss: 1.9489: 100%|██████████| 255/255 [00:23<00:00, 10.81it/s]


Another mode is *stateful* mode, where the input and hidden spike functions preserve their kernel states between forward calls. This lets us split a sequence into smaller blocks when the full sequence does not fit in memory.

The loop below processes each block with `StepMode.MULTI_STATEFUL` and backpropagates its loss immediately. We weight each block's mean loss by its fraction of the sequence, including a shorter final block. After each backward pass, `model.detach()` detaches the saved state while preserving its values for the next block. Reset the model only at the start of each batch and update the weights once after all blocks.

In [4]:
step_mode = StepMode.MULTI_STATEFUL
T_per_split = 10
model.train()
n_epochs = 1

for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        total_loss = 0
        for data_split in data.split(T_per_split, dim=0):
            y = model(data_split, step_mode=step_mode)
            loss = loss_fn(y, labels) * (data_split.shape[0] / data.shape[0])
            loss.backward()
            total_loss += loss.item()
            model.detach()

        optimizer.step()
        pbar.set_description(f"Loss: {total_loss:.4f}")

Loss: 1.1123: 100%|██████████| 255/255 [00:29<00:00,  8.66it/s]


## Testing
Pass the full sequence in multi-step mode, then sum the continuous output signal over time to compute predictions, as in the previous notebooks.

In [5]:
step_mode = StepMode.MULTI_STATELESS
batch_iter = iter(test_dataloader)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    pbar = tqdm(batch_iter, total=len(test_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        y = model(data, step_mode=step_mode)
        predictions = y.sum(dim=0).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

100%|██████████| 18/18 [00:02<00:00,  7.40it/s]

Test Accuracy: 77.87%
